In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

In [13]:
df = pd.read_csv('heart.csv')

In [14]:
# Separate features (X) and target (y)
X = df.drop('target', axis=1)
y = df['target']

In [15]:
# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# Apply StandardScaler
# This ensures mean=0 and variance=1, which is crucial for both LR and HE
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
# --------------------------------------------------
# Degree-5 sigmoid approximation
# P(z) = 0.5 + a1*z + a3*z^3 + a5*z^5
# --------------------------------------------------

def sigmoid_poly(z):
    return (
        0.5
        + 0.185061021 * z
        - 0.00404514417 * (z ** 3)
        + 0.0000333077433 * (z ** 5)
    )

In [ ]:
# --------------------------------------------------
# Plaintext Logistic Regression training
# Mirrors the HELR encrypted-training structure
# --------------------------------------------------

def train_plain_he_lr(
    X,
    y,
    epochs=5,
    learning_rate=1.0,
    reg_lambda=0.059
):
    n_samples, n_features = X.shape

    # Paper's Algorithm 3 initializes encrypted W from zero.
    # Here we use plaintext zero weights as the reference.
    W = np.zeros(n_features, dtype=np.float64)

    loss_history = []
    accuracy_history = []

    for epoch in range(epochs):

        # ------------------------------------------
        # 1. Forward pass
        # z = XW
        # ------------------------------------------
        z = X @ W

        # ------------------------------------------
        # 2. Approximate sigmoid
        # p ≈ sigmoid(z)
        # ------------------------------------------
        p = sigmoid_poly(z)

        # ------------------------------------------
        # 3. Gradient
        # g = X^T(p-y)/N + lambda*W
        # ------------------------------------------
        error = p - y

        gradient = (
            (X.T @ error) / n_samples
            + reg_lambda * W
        )

        # ------------------------------------------
        # 4. Weight update
        # ------------------------------------------
        W = W - learning_rate * gradient

        # ------------------------------------------
        # Monitoring
        # ------------------------------------------
        predictions = (p >= 0.5).astype(int)
        accuracy = np.mean(predictions == y)

        # We will use ordinary BCE only as a monitoring
        # metric. The model itself uses polynomial sigmoid.
        eps = 1e-12
        p_clip = np.clip(p, eps, 1 - eps)

        loss = -np.mean(
            y * np.log(p_clip)
            + (1 - y) * np.log(1 - p_clip)
        )

        loss_history.append(loss)
        accuracy_history.append(accuracy)

        print(
            f"Epoch {epoch + 1:2d} | "
            f"Loss = {loss:.6f} | "
            f"Accuracy = {accuracy * 100:.2f}%"
        )

    return W, loss_history, accuracy_history

In [22]:
for epochs in [5, 10, 20, 50]:
    print(f"\n========== {epochs} EPOCHS ==========")

    W_test, loss_history, accuracy_history = train_plain_he_lr(
        X_train_scaled,
        y_train.values,
        epochs=epochs,
        learning_rate=1.0,
        reg_lambda=0.059
    )

    z_test = X_test_scaled @ W_test
    p_test = sigmoid_poly(z_test)
    y_pred = (p_test >= 0.5).astype(int)

    acc = np.mean(y_pred == y_test.values)

    print(f"Test Accuracy = {acc * 100:.2f}%")


========== 5 EPOCHS ==========
Epoch  1 | Loss = 0.693147 | Accuracy = 51.59%
Epoch  2 | Loss = 0.496990 | Accuracy = 84.15%
Epoch  3 | Loss = 0.434569 | Accuracy = 83.90%
Epoch  4 | Loss = 0.406880 | Accuracy = 85.37%
Epoch  5 | Loss = 0.392064 | Accuracy = 85.61%
Test Accuracy = 79.02%

========== 10 EPOCHS ==========
Epoch  1 | Loss = 0.693147 | Accuracy = 51.59%
Epoch  2 | Loss = 0.496990 | Accuracy = 84.15%
Epoch  3 | Loss = 0.434569 | Accuracy = 83.90%
Epoch  4 | Loss = 0.406880 | Accuracy = 85.37%
Epoch  5 | Loss = 0.392064 | Accuracy = 85.61%
Epoch  6 | Loss = 0.383209 | Accuracy = 86.10%
Epoch  7 | Loss = 0.377606 | Accuracy = 86.10%
Epoch  8 | Loss = 0.373860 | Accuracy = 86.10%
Epoch  9 | Loss = 0.371246 | Accuracy = 85.73%
Epoch 10 | Loss = 0.369374 | Accuracy = 85.37%
Test Accuracy = 79.02%

========== 20 EPOCHS ==========
Epoch  1 | Loss = 0.693147 | Accuracy = 51.59%
Epoch  2 | Loss = 0.496990 | Accuracy = 84.15%
Epoch  3 | Loss = 0.434569 | Accuracy = 83.90%
Epoch  4 |

In [59]:
import tenseal as ts

context_train = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[
        40, 21, 21, 21, 21, 21, 21, 40
    ]
)

context_train.global_scale = 2**21

context_train.generate_galois_keys()
context_train.generate_relin_keys()

print("CKKS training context created.")

CKKS training context created.


In [60]:
x0 = X_train_scaled[0]

enc_x0 = ts.ckks_vector(
    context_train,
    x0.tolist()
)

print("Original:")
print(x0)

print("\nDecrypted:")
print(enc_x0.decrypt())

Original:
[-0.58584022  0.65465367  1.008275   -0.77945357 -1.93503098 -0.41403934
 -0.9837419  -1.01909426 -0.72594894 -0.21066121  1.00526437  2.17169136
 -0.54519316]

Decrypted:
[-0.5857208763342999, 0.653558592467462, 1.0082320693572393, -0.7795118628633464, -1.9354487814296228, -0.4129812484863805, -0.983835748302721, -1.019615817898633, -0.7255691110325362, -0.21050760473045588, 1.004686889520784, 2.1710540268275818, -0.5453824037309818]


In [61]:
y0 = float(y_train.iloc[0])

enc_y0 = ts.ckks_vector(
    context_train,
    [y0]
)

print("Label:", y0)
print("Decrypted label:", enc_y0.decrypt())

Label: 0.0
Decrypted label: [0.0005593519375782811]


In [62]:
W0 = np.zeros(X_train_scaled.shape[1])

enc_W0 = ts.ckks_vector(
    context_train,
    W0.tolist()
)

print("Initial encrypted weights:")
print(enc_W0.decrypt())

Initial encrypted weights:
[0.00015867512280432123, 0.0013543136978390981, 0.00029912654968323777, -0.0004022781831784755, -0.0007326563984141733, -0.0008098328708978742, -7.216508532518402e-05, 0.0008541929021579134, -0.00026368482713516584, -0.00044295743479939187, -0.000522132780417865, 0.0004851717137706997, 9.102082786414682e-05]


In [63]:
# Encrypted forward pass for one sample

enc_z0 = enc_x0.dot(enc_W0)

z0_he = enc_z0.decrypt()[0]

print("Encrypted z decrypted:", z0_he)

Encrypted z decrypted: -0.7717710110262446


In [64]:
# Test encrypted-encrypted element-wise multiplication

enc_mul = enc_x0 * enc_W0

mul_result = enc_mul.decrypt()

print("X:", enc_x0.decrypt())
print("W:", enc_W0.decrypt())
print("\nElement-wise HE product:")
print(mul_result)

X: [-0.5857208763342999, 0.653558592467462, 1.0082320693572393, -0.7795118628633464, -1.9354487814296228, -0.4129812484863805, -0.983835748302721, -1.019615817898633, -0.7255691110325362, -0.21050760473045588, 1.004686889520784, 2.1710540268275818, -0.5453824037309818]
W: [0.00015867512280432123, 0.0013543136978390981, 0.00029912654968323777, -0.0004022781831784755, -0.0007326563984141733, -0.0008098328708978742, -7.216508532518402e-05, 0.0008541929021579134, -0.00026368482713516584, -0.00044295743479939187, -0.000522132780417865, 0.0004851717137706997, 9.102082786414682e-05]

Element-wise HE product:
[0.00016042968400409167, 0.002064905228974992, 0.0005445967043266849, 0.001976823354957586, 0.0019110970312685939, -0.0010216949268101075, -0.001222056814992524, -0.00224912681210209, 0.0002899168062599051, -0.000579652391380534, -0.0015229334752414983, 0.003222179919993487, 0.00015757791566445935]


In [65]:
expected_sum = np.dot(
    np.array(enc_x0.decrypt()),
    np.array(enc_W0.decrypt())
)

print("Dot of decrypted X and W:", expected_sum)
print("Sum of HE products:", sum(mul_result))

Dot of decrypted X and W: 0.0031235486981687267
Sum of HE products: 0.003732062224923046


In [66]:
enc_sum = enc_mul.sum()

print("HE sum:", enc_sum.decrypt())

HE sum: [-0.7717710110262446]


In [67]:
a = [1.0, 2.0, 3.0, 4.0]
b = [0.5, 1.0, 1.5, 2.0]

enc_a = ts.ckks_vector(context_train, a)
enc_b = ts.ckks_vector(context_train, b)

enc_product = enc_a * enc_b

print("Expected:", [0.5, 2.0, 4.5, 8.0])
print("HE      :", enc_product.decrypt()[:4])

Expected: [0.5, 2.0, 4.5, 8.0]
HE      : [0.5860031670405217, 2.3527153605322995, 5.2835345525088435, 9.395851342906607]


In [68]:
enc_dot = enc_a.dot(enc_b)

print("Expected dot:", 15.5)
print("HE dot      :", enc_dot.decrypt()[0])

Expected dot: 15.5
HE dot      : 17.479450972538267


In [69]:
print("Global scale:", context_train.global_scale)
print("Auto relin:", context_train.auto_relin)
print("Auto rescale:", context_train.auto_rescale)
print("Auto mod switch:", context_train.auto_mod_switch)

Global scale: 2097152.0
Auto relin: True
Auto rescale: True
Auto mod switch: True


In [70]:
test_context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)

test_context.global_scale = 2**40

test_context.generate_galois_keys()
test_context.generate_relin_keys()

In [71]:
a = [1.0, 2.0, 3.0, 4.0]
b = [0.5, 1.0, 1.5, 2.0]

enc_a = ts.ckks_vector(test_context, a)
enc_b = ts.ckks_vector(test_context, b)

enc_product = enc_a * enc_b

print("Expected product:", [0.5, 2.0, 4.5, 8.0])
print("HE product      :", enc_product.decrypt()[:4])

enc_dot = enc_a.dot(enc_b)

print("Expected dot:", 15.0)
print("HE dot      :", enc_dot.decrypt()[0])

Expected product: [0.5, 2.0, 4.5, 8.0]
HE product      : [0.5000000657801738, 2.0000002667002614, 4.500000609030097, 8.000001065867126]
Expected dot: 15.0
HE dot      : 15.0000023983679


In [72]:
x = [1.0, 2.0, 3.0, 4.0]
w = [0.1, 0.2, 0.3, 0.4]

enc_x = ts.ckks_vector(test_context, x)
enc_w = ts.ckks_vector(test_context, w)

enc_z = enc_x.dot(enc_w)

z_he = enc_z.decrypt()[0]

z_plain = np.dot(x, w)

print("Plain z:", z_plain)
print("HE z   :", z_he)
print("Error  :", abs(z_plain - z_he))

Plain z: 3.0
HE z   : 3.0000008100373985
Error  : 8.100373984909481e-07


In [73]:
coeffs5_wide = [
    0.5,
    0.185061021,
    0.0,
    -0.00404514417,
    0.0,
    0.0000333077433
]

enc_p = enc_z.polyval(coeffs5_wide)

p_he = enc_p.decrypt()[0]

p_plain = np.polyval(
    coeffs5_wide[::-1],
    np.dot(x, w)
)

print("Plain polynomial:", p_plain)
print("HE polynomial   :", p_he)
print("CKKS error      :", abs(p_plain - p_he))

ValueError: scale out of bounds